In [13]:
from transformers import BertTokenizerFast, BertModel
import spacy
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import os
import numpy as np

In [14]:
from data import *
from model import *
from utilsv2 import *

In [15]:
os.environ["TOKENIZERS_PARALLELISM"] = "true"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Khởi tạo tokenizer và model BERT
tokenizer_bert = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")

tokenizer_bert = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")

# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer_bert.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer_bert))  # Resize embedding BERT


# Tokenizer và DataLoader
spacy_nlp = spacy.load('en_core_web_sm')
spacy_nlp.tokenizer = create_custom_tokenizer(spacy_nlp, special_tokens['additional_special_tokens'])

In [16]:
data_source = "/data/ddao/TRE/data/raw_data/i2b2_2012/data_processed/"
data_train_path = data_source + "train_merged_full.csv"
data_test_path = data_source + "test_merged_full.csv"


train_df = pd.read_csv(data_train_path)
test_df = pd.read_csv(data_test_path)

# Mã hóa nhãn
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['label'])
test_df['label_encoded'] = label_encoder.transform(test_df['label'])  # Ánh xạ giống tập huấn luyện
num_classes = len(label_encoder.classes_)
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
id2label = {v: k for k, v in label_mapping.items()}
print(f"Label Mapping: {label_mapping}")
print(f"id2label: {id2label}")
relations = list(label_mapping.keys())


validation_size = 0.15
random_state = 42
train_df, valid_df = train_test_split(train_df, test_size=validation_size, random_state=random_state)

print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("test_df:", test_df.shape)

Label Mapping: {'AFTER': 0, 'BEFORE': 1, 'OVERLAP': 2}
id2label: {0: 'AFTER', 1: 'BEFORE', 2: 'OVERLAP'}
train_df: (28305, 12)
valid_df: (4995, 12)
test_df: (27530, 12)


In [17]:
test_df

,entity1_id,entity2_id,entity1_start,entity2_start,entity1_end,entity2_end,entity1_text,entity2_text,document_id,text,label,label_encoded
0,E0,T0,1,18,10,28,Admission,2012-06-07,88,\nAdmission Date :\n2012-06-07\nDischarge Date...,OVERLAP,2
1,E1,T1,29,46,38,56,Discharge,2012-06-09,88,\nAdmission Date :\n2012-06-07\nDischarge Date...,OVERLAP,2
2,E10,E8,387,338,394,343,Inderal,pills,88,\nAdmission Date :\n2012-06-07\nDischarge Date...,OVERLAP,2
3,E11,E10,399,387,407,394,Klonopin,Inderal,88,\nAdmission Date :\n2012-06-07\nDischarge Date...,OVERLAP,2
4,E12,E54,460,1498,466,1504,geodon,Geodon,88,\nAdmission Date :\n2012-06-07\nDischarge Date...,OVERLAP,2
...,...,...,...,...,...,...,...,...,...,...,...,...
27525,E28,T0,820,18,839,28,an outside hospital,12/22/2000,46,\nAdmission Date :\n12/22/2000\nDischarge Date...,BEFORE,1
27526,E29,T0,854,18,859,28,noted,12/22/2000,46,\nAdmission Date :\n12/22/2000\nDischarge Date...,BEFORE,1
27527,E30,T0,879,18,890,28,discrepancy,12/22/2000,46,\nAdmission Date :\n12/22/2000\nDischarge Date...,BEFORE,1
27528,E31,T0,894,18,929,28,her upper extremity blood pressures,12/22/2000,46,\nAdmission Date :\n12/22/2000\nDischarge Date...,BEFORE,1


In [18]:
print("Create dataloader:")
batch_size = 128
window_size = 128
MAX_LENGTH = 256
shuffle=False
print("batch_size: ", batch_size)
print("window_size: ", window_size)
print("MAX_LENGTH: ", MAX_LENGTH)

Create dataloader:
batch_size:  128
window_size:  128
MAX_LENGTH:  256


In [19]:
# train_dataloader = create_dataloader(train_df, spacy_nlp, tokenizer_bert, special_tokens, batch_size, window_size, MAX_LENGTH, shuffle, is_I2B2=True)
# validation_dataloader = create_dataloader(valid_df, spacy_nlp, tokenizer_bert, special_tokens, batch_size, window_size, MAX_LENGTH, shuffle, is_I2B2=True)
# test_dataloader = create_dataloader(test_df, spacy_nlp, tokenizer_bert, special_tokens, batch_size, window_size, MAX_LENGTH, shuffle, is_I2B2=True)

In [20]:
# test_dataloader_fast = create_dataloader(test_df, spacy_nlp, tokenizer_bert, special_tokens, 512, window_size, MAX_LENGTH, shuffle, is_I2B2=True)

In [21]:
train_dataloader_path =  f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/train_dataloader_ws_{window_size}.pt"
validation_dataloader_path = f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/validation_dataloader_ws_{window_size}.pt"
test_dataloader_path = f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/test_dataloader_ws_{window_size}.pt"

In [22]:
# torch.save(train_dataloader, train_dataloader_path)
# torch.save(validation_dataloader, validation_dataloader_path)
# torch.save(test_dataloader, test_dataloader_path)

In [23]:
# train_dataloader = torch.load(train_dataloader_path)
# validation_dataloader = torch.load(validation_dataloader_path)
# test_dataloader =. torch.load(test_dataloader_path)

In [24]:
test_dataloader_fast_path = f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/test_dataloader_ws_{512}.pt"
# torch.save(test_dataloader_fast, test_dataloader_fast_path)
test_dataloader = torch.load(test_dataloader_fast_path)

In [25]:
# ============================================================
# Hyperparameters
# ============================================================
hidden_ffn_dim   = 512
num_heads        = 8
num_layers       = 1
dropout          = 0.3
num_classes      = len(label_encoder.classes_)

num_epochs       = 30
eval_step        = 1
LEARNING_RATE    = 1e-5

top_k            = 10

print(f"hidden_ffn_dim: {hidden_ffn_dim}")
print(f"num_layers: {num_layers}")
print(f"num_heads: {num_heads}")
print(f"top_k: {top_k}")

# ============================================================
# Save path
# ============================================================

save_path = (
    f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/"
    f"FTBERT_I2B2_MHA_{num_layers}_FFN_{hidden_ffn_dim}_TOP_{top_k}.pth"
)
print('Model will save at:', save_path)

hidden_ffn_dim: 512
num_layers: 1
num_heads: 8
top_k: 10
Model will save at: /data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/FTBERT_I2B2_MHA_1_FFN_512_TOP_10.pth


In [26]:
print("\nCreate model:")
model = TemporalRelationModelV2(
    bert=bert,
    tokenizer=tokenizer_bert,
    hidden_ffn_dim=hidden_ffn_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout,
    num_classes=num_classes,
    device=device
).to(device)

print("Device:", device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("--" * 20)


Create model:
Device: cuda
----------------------------------------


/home/prof/ddao/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [27]:
print("\nStart Training...")
losses = train_model(
    model=model,
    top_k=top_k,
    train_dataloader=train_dataloader,
    validation_dataloader=validation_dataloader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=num_epochs,
    save_path=save_path,
    eval_step=eval_step,
    max_grad_norm=1.0,      # clip grad
    lambda_aux=1.0,         # weight cho aux loss (0 nếu không dùng)
    is_I2B2=True
)


Start Training...


NameError: name 'train_dataloader' is not defined

# Evaluate

In [28]:
print("Load pretrained model:")
checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']
print(f"Model saved at epoch {epoch}, loss = {loss:.4f}")

Load pretrained model:
Model saved at epoch 28, loss = 0.0381


In [ ]:
def grid_search(data_loader, model, K):
    metrics_by_topk = []
    for k in range(0,K+1,1):
        cm, f1, p, r, precisions,recalls, f1s = evaluate(data_loader,
                                                        model=model,
                                                        top_k=k,
                                                        device=device,
                                                        return_preds=False,
                                                        is_I2B2=True)
        metrics_by_topk.append([k, f1, p, r])
        print(f"top_K: {k} - F1: {f1:.4f} | Precision: {p:.4f} | Recall: {r:.4f}")
        print()
    return metrics_by_topk

In [ ]:
metrics_by_topk = grip_search(test_dataloader, model, 20)

In [20]:
# Tạo DataFrame
df = pd.DataFrame(metrics_by_topk, columns=["top_k", "F1", "Precision", "Recall"])
df.to_csv("/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/metrics_by_topk.csv", index=False, encoding='utf-8-sig')

In [29]:
df = pd.read_csv("/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/metrics_by_topk.csv", encoding='utf-8-sig')

In [30]:
# Lấy hàng có F1 cao nhất
best_row = df.loc[df['F1'].idxmax()]

best_k = int(best_row['top_k'])
best_f1 = best_row['F1']
best_p = best_row['Precision']
best_r = best_row['Recall']

print(f"✅ Top-K tốt nhất: {best_k}")
print(f"F1 = {best_f1:.4f} | Precision = {best_p:.4f} | Recall = {best_r:.4f}")


✅ Top-K tốt nhất: 5
F1 = 0.8692 | Precision = 0.8718 | Recall = 0.8717


In [31]:
cm, f1, p, r, precisions,recalls, f1s = evaluate(test_dataloader,
                                                        model=model,
                                                        top_k=best_k,
                                                        device=device,
                                                        return_preds=False,
                                                        is_I2B2=True,
                                                        test_df=test_df,
                                                        id2label=id2label)

Evaluating: 100%|██████████| 54/54 [01:57<00:00,  2.17s/it]


Evaluation finished in 117.15s.
Using closure-based evaluation for I2B2 (document-level).
====== [I2B2 CLOSURE-BASED EVALUATION] ======
Precision: 0.8726
Recall:    0.8742
F1-score:  0.8734
n_gold:  27530
n_pred:  27530
n_gold_closure:  175550
n_pred_closure:  135866


In [27]:
torch.cuda.empty_cache()